## Extension A: A three-node network

In [1]:
import numpy as np
from scipy.optimize import root

# --- Parameters for Extension A (3 Nodes) ---
L_bar = np.array([100.0, 100.0, 100.0])
A = np.array([1.0, 1.0, 1.0])
beta = np.array([
    [0.6, 0.2, 0.2],
    [0.2, 0.6, 0.2],
    [0.2, 0.2, 0.6]
])
eta_H = 4.0

# Base transport costs (3x3 matrix)
t_base = np.array([
    [0.0, 0.2, 0.2],
    [0.2, 0.0, 0.2],
    [0.2, 0.2, 0.0]
])

# Improved transport link between Node 1 and Node 2
t_improved = np.array([
    [0.0, 0.05, 0.2],
    [0.05, 0.0, 0.2],
    [0.2,  0.2, 0.0]
])

def price_ij(t, w, A):
    p_factory = w / A
    return p_factory[:, np.newaxis] * (1 + t)

def household_price_index(p_ij, beta, eta_H):
    return (np.sum(beta * (p_ij ** (1 - eta_H)), axis=0)) ** (1 / (1 - eta_H))

def armington_shares(p_ij, beta, eta_H):
    return (beta * (p_ij ** (1 - eta_H))) / np.sum(beta * (p_ij ** (1 - eta_H)), axis=0)

def income(w, L_bar):
    return w * L_bar

def value_market_clearing(t, w, L_bar, beta, A, eta_H):
    p_delivered = price_ij(t, w, A)
    s_H = armington_shares(p_delivered, beta, eta_H)
    I = income(w, L_bar)
    
    # Revenue = sum over destinations j of expenditure shares * income
    revenue = np.sum(s_H * I, axis=1)
    supply = w * L_bar
    return revenue - supply

def solve_extension_a(t_matrix):
    # Solver for w2 and w3 with numeraire w1 = 1
    def residuals(x):
        w = np.array([1.0, np.exp(x[0]), np.exp(x[1])])
        val_clearing = value_market_clearing(t_matrix, w, L_bar, beta, A, eta_H)
        return val_clearing[1:] # Return market residuals for nodes 2 and 3

    res = root(residuals, [0.0, 0.0])
    w_sol = np.array([1.0, np.exp(res.x[0]), np.exp(res.x[1])])
    
    p_ij_sol = price_ij(t_matrix, w_sol, A)
    P_H_sol = household_price_index(p_ij_sol, beta, eta_H)
    I_sol = income(w_sol, L_bar)
    Q_sol = I_sol / P_H_sol
    
    return w_sol, P_H_sol, Q_sol

# --- Run comparison ---
w_b, P_b, Q_b = solve_extension_a(t_base)
w_i, P_i, Q_i = solve_extension_a(t_improved)

print("=== BASELINE RESULTS (t12 = 0.20) ===")
print("Wages (w1, w2, w3):", w_b)
print("Household Price Index (P_H):", P_b)
print("Real Consumption (Q):", Q_b)

print("\n=== IMPROVED LINK 1-2 (t12 = 0.05) ===")
print("Wages (w1, w2, w3):", w_i)
print("Household Price Index (P_H):", P_i)
print("Real Consumption (Q):", Q_i)

=== BASELINE RESULTS (t12 = 0.20) ===
Wages (w1, w2, w3): [1. 1. 1.]
Household Price Index (P_H): [1.06344689 1.06344689 1.06344689]
Real Consumption (Q): [94.03384482 94.03384482 94.03384482]

=== IMPROVED LINK 1-2 (t12 = 0.05) ===
Wages (w1, w2, w3): [1.         1.         0.98858591]
Household Price Index (P_H): [1.03861261 1.03861261 1.05463222]
Real Consumption (Q): [96.28228937 96.28228937 93.73750374]


Even though transport links connected to node 3 remain unchanged, node 3 is indirectly affected by the improvement in link 1-2. Lowering transport costs between nodes 1 and 2 lowers prices and increases real income in nodes 1 and 2, which shifts trade demand toward each other and away from node 3 (trade diversion). As a result, node 3 loses welfare as its real consumption falls from 94.03 to 93.73 even though its own direct transport links remain unchanged. This outcome is driven by all three factors working together: trade diversion causes nodes 1 and 2 to buy more from each other and reduce their demand for goods from node 3, this drop in demand forces node 3's wage down from 1.0 to 0.9886, and although its production costs and overall price index drop slightly from 1.0634 to 1.0546, this minor price reduction is not enough to offset the loss in household wage income.

### Extension B: CES Production Analysis

In [2]:
import numpy as np
from scipy.optimize import root

# --- Base Parameters ---
N = 2
L_bar = np.array([100.0, 100.0])
phi = np.array([1.0, 1.0])
delta_l = 0.6

# Intermediate input weights delta_ij (rows = origin, columns = destination)
# delta_lj + sum_i(delta_ij) = 0.6 + (0.24 + 0.16) = 1.0
delta_ij = np.array([
    [0.6 * 0.4, 0.4 * 0.4],  # Origin 1 -> Destination 1, Destination 2
    [0.4 * 0.4, 0.6 * 0.4]   # Origin 2 -> Destination 1, Destination 2
])

beta = np.array([[0.6, 0.4], [0.4, 0.6]])
t = np.array([[0.0, 0.2], [0.2, 0.0]])
eta_H = 4.0

def price_ij(t, p):
    return p[:, np.newaxis] * (1.0 + t)

def household_price_index(p_ij, beta, eta_H):
    return (np.sum(beta * (p_ij ** (1.0 - eta_H)), axis=0)) ** (1.0 / (1.0 - eta_H))

def armington_shares(p_ij, beta, eta_H):
    return (beta * (p_ij ** (1.0 - eta_H))) / np.sum(beta * (p_ij ** (1.0 - eta_H)), axis=0)

def income(w, L_bar):
    return w * L_bar

def solve_model_ces(sigma_val, t_mat):
    def system_residuals(x):
        p = np.exp(x[:2])
        w = np.array([1.0, np.exp(x[2])])
        
        p_ij = price_ij(t_mat, p)
        
        # 1. Standardized CES unit cost (Equation 89)
        if np.isclose(sigma_val, 1.0):
            prod_p = np.prod(p_ij ** delta_ij, axis=0)
            c = (1.0 / phi) * (w ** delta_l) * prod_p
        else:
            term_l = delta_l * (w ** (1.0 - sigma_val))
            term_x = np.sum(delta_ij * (p_ij ** (1.0 - sigma_val)), axis=0)
            c = (1.0 / phi) * ((term_l + term_x) ** (1.0 / (1.0 - sigma_val)))
            
        R_profit = p - c
        
        # 2. Input cost shares (M_ij) and labor share (theta_L)
        if np.isclose(sigma_val, 1.0):
            M = delta_ij
            theta_L = delta_l
        else:
            M = delta_ij * (p_ij / (phi * c)) ** (1.0 - sigma_val)
            theta_L = delta_l * (w / (phi * c)) ** (1.0 - sigma_val)
            
        s_H = armington_shares(p_ij, beta, eta_H)
        I = income(w, L_bar)
        b = np.sum(s_H * I, axis=1)
        
        # Production value V = (I - M)^(-1) * b
        V = np.linalg.solve(np.eye(N) - M, b)
        
        # Labor demand
        L_d = theta_L * V / w
        R_labor = L_d[1] - L_bar[1]
        
        return np.array([R_profit[0], R_profit[1], R_labor])

    # Starts strictly from the symmetric equilibrium log(p1)=0, log(p2)=0, log(w2)=0
    sol = root(system_residuals, [0.0, 0.0, 0.0], method='hybr')
    
    p_sol = np.exp(sol.x[:2])
    w_sol = np.array([1.0, np.exp(sol.x[2])])
    
    p_ij_sol = price_ij(t_mat, p_sol)
    P_H_sol = household_price_index(p_ij_sol, beta, eta_H)
    I_sol = income(w_sol, L_bar)
    Q_sol = I_sol / P_H_sol
    
    return w_sol, p_sol, P_H_sol, Q_sol

# --- Execution ---
for sig in [0.5, 1.0, 2.0]:
    w_s, p_s, P_s, Q_s = solve_model_ces(sig, t)
    print(f"=== CES RESULTS with sigma = {sig} ===")
    print("Wages (w):", np.round(w_s, 4))
    print("Factory prices (p):", np.round(p_s, 4))
    print("Price Index (P_H):", np.round(P_s, 4))
    print("Real Consumption / Welfare (Q):", np.round(Q_s, 4))
    print("-" * 45)

=== CES RESULTS with sigma = 0.5 ===
Wages (w): [1. 1.]
Factory prices (p): [1.0529 1.0529]
Price Index (P_H): [1.1197 1.1197]
Real Consumption / Welfare (Q): [89.3081 89.3081]
---------------------------------------------
=== CES RESULTS with sigma = 1.0 ===
Wages (w): [1. 1.]
Factory prices (p): [1.0498 1.0498]
Price Index (P_H): [1.1164 1.1164]
Real Consumption / Welfare (Q): [89.5714 89.5714]
---------------------------------------------
=== CES RESULTS with sigma = 2.0 ===
Wages (w): [1. 1.]
Factory prices (p): [1.0444 1.0444]
Price Index (P_H): [1.1107 1.1107]
Real Consumption / Welfare (Q): [90.0324 90.0324]
---------------------------------------------


Replacing Cobb-Douglas production with a CES technology introduces an additional margin of behavioral adjustment. When transport costs increase delivered input prices, a higher elasticity of substitution ($\sigma_j > 1$) enables firms to adjust their factor input mix away from more expensive inputs[cite: 31].

As shown by the numerical results:
1. **Lower Elasticity ($\sigma = 0.5$):** Rigid factor proportions limit optimization, resulting in higher factory-gate prices ($p_j = 1.0529$) and lower real consumption ($Q_j = 89.3081$).
2. **Benchmark Cobb-Douglas ($\sigma = 1.0$):** Yields standard general equilibrium baseline values ($p_j = 1.0498$, $Q_j = 89.5714$).
3. **Higher Elasticity ($\sigma = 2.0$):** Greater input flexibility dampens the transport-cost shock, lowering unit production costs ($p_j = 1.0444$) and improving real welfare ($Q_j = 90.0324$).

## Extension C: Equivalent and compensating variation

In [3]:
import numpy as np
from scipy.optimize import root

# --- Part VI / Model 5 Parameters ---
N = 2
L_bar = np.array([100.0, 100.0])
phi = np.array([1.0, 1.0])
delta_l = np.array([0.6, 0.6])
delta_x = np.array([0.4, 0.4])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])
gamma = np.array([[0.6, 0.4], [0.4, 0.6]])
eta_H = 4.0
eta_X = 4.0

# Transport cost shock (from 0.20 to 0.10)
t_0 = np.array([[0.0, 0.20], [0.20, 0.0]])
t_1 = np.array([[0.0, 0.10], [0.10, 0.0]])

def price_ij(t, p):
    return p[:, np.newaxis] * (1.0 + t)

def household_price_index(p_ij, beta, eta_H):
    return (np.sum(beta * (p_ij ** (1.0 - eta_H)), axis=0)) ** (1.0 / (1.0 - eta_H))

def intermediate_price_index(p_ij, gamma, eta_X):
    return (np.sum(gamma * (p_ij ** (1.0 - eta_X)), axis=0)) ** (1.0 / (1.0 - eta_X))

def armington_shares(p_ij, share_mat, eta):
    return (share_mat * (p_ij ** (1.0 - eta))) / np.sum(share_mat * (p_ij ** (1.0 - eta)), axis=0)

def solve_part5_equilibrium(t_mat):
    def system_residuals(x):
        p = np.exp(x[:2])
        w = np.array([1.0, np.exp(x[2])])
        p_ij = price_ij(t_mat, p)
        P_X = intermediate_price_index(p_ij, gamma, eta_X)
        
        c = (1.0 / phi) * (w ** delta_l) * (P_X ** delta_x)
        R_profit = p - c
        
        s_X = armington_shares(p_ij, gamma, eta_X)
        M = s_X * delta_x
        
        s_H = armington_shares(p_ij, beta, eta_H)
        I = w * L_bar
        b = np.sum(s_H * I, axis=1)
        
        V = np.linalg.solve(np.eye(N) - M, b)
        L_d = delta_l * V / w
        R_labor = L_d[1] - L_bar[1]
        
        return np.array([R_profit[0], R_profit[1], R_labor])

    sol = root(system_residuals, [0.0, 0.0, 0.0])
    p_sol = np.exp(sol.x[:2])
    w_sol = np.array([1.0, np.exp(sol.x[2])])
    
    p_ij_sol = price_ij(t_mat, p_sol)
    P_H_sol = household_price_index(p_ij_sol, beta, eta_H)
    I_sol = w_sol * L_bar
    Q_sol = I_sol / P_H_sol
    
    return w_sol, p_sol, P_H_sol, I_sol, Q_sol

# 1. Solve Equilibria
w0, p0, P0_H, I0, Q0 = solve_part5_equilibrium(t_0)
w1, p1, P1_H, I1, Q1 = solve_part5_equilibrium(t_1)

# 2. Calculate Monetary Welfare Measures (Extension C)
EV = P0_H * (Q1 - Q0)  # P0_H * Q1 - I0
CV = P1_H * (Q1 - Q0)  # I1 - P1_H * Q0
pct_change_Q = ((Q1 - Q0) / Q0) * 100.0

print("=== INITIAL EQUILIBRIUM (t = 0.20) ===")
print("Price Index (P_H0):", np.round(P0_H, 4))
print("Real Consumption (Q0):", np.round(Q0, 4))

print("\n=== NEW EQUILIBRIUM (t = 0.10) ===")
print("Price Index (P_H1):", np.round(P1_H, 4))
print("Real Consumption (Q1):", np.round(Q1, 4))

print("\n=== EXTENSION C: MONETARY WELFARE MEASURES ===")
print("Equivalent Variation (EV_j):", np.round(EV, 4))
print("Compensating Variation (CV_j):", np.round(CV, 4))

=== INITIAL EQUILIBRIUM (t = 0.20) ===
Price Index (P_H0): [1.108 1.108]
Real Consumption (Q0): [90.2555 90.2555]

=== NEW EQUILIBRIUM (t = 0.10) ===
Price Index (P_H1): [1.0599 1.0599]
Real Consumption (Q1): [94.3453 94.3453]

=== EXTENSION C: MONETARY WELFARE MEASURES ===
Equivalent Variation (EV_j): [4.5313 4.5313]
Compensating Variation (CV_j): [4.3349 4.3349]


All three measures ($EV_j$, $CV_j$, and $\% \Delta Q_j$) are positive, showing that lowering transport costs clearly improves welfare in both regions.

* **$EV_j = 4.53$** uses old prices ($P^{H,0} = 1.1080$). It matches the $4.53\%$ rise in real consumption because initial income is $100$.
* **$CV_j = 4.33$** uses new prices ($P^{H,1} = 1.0599$). Because prices are lower now, you need less nominal money to get the same utility gain, so $CV < EV$.